# Spark Cluster Run: Banking77 Fraud Stream (full profile)

This notebook runs on the **master EC2 instance** and uses the 3-worker standalone Spark cluster.

1.  Connect to the cluster (same setup as HW7)
2.  Read the raw simulated stream from S3 (many small files) and time it
3.  Compact the stream to one Parquet file per day and write it back to S3
4.  Re-read the compacted data and time it again
5.  Run the Part C aggregations (overview, message gaps, sessions) on the full stream
6.  Save small result tables for the Spark 1-pager

Only small aggregated results come back to the driver; the 50M-row stream stays on the cluster.

In [1]:
import time
from pathlib import Path

import pandas as pd
from pyspark.sql import SparkSession, Window
from pyspark.sql import functions as F

# ---- Settings ----
BUCKET = "dsan6000-banking77-mh"
LOCAL_TEST = False                 # True only for a laptop test with local files

if LOCAL_TEST:
    MASTER = "local[*]"
    ROOT = Path.cwd().parent if Path.cwd().name == "spark" else Path.cwd()
    RAW = (ROOT / "data" / "simulated").as_posix()
    OUT = (ROOT / "data" / "cluster_test").as_posix()
else:
    master_private_ip = !hostname -I
    MASTER = f"spark://{master_private_ip[0].split()[0]}:7077"
    RAW = f"s3a://{BUCKET}/banking77/simulated"          # uploaded with aws s3 sync
    OUT = f"s3a://{BUCKET}/banking77"                    # compacted data and results go here

LOCAL_OUT = Path("output")
LOCAL_OUT.mkdir(exist_ok=True)
print("master:", MASTER)
print("raw data:", RAW)

master: spark://172.31.46.151:7077
raw data: s3a://dsan6000-banking77-mh/banking77/simulated

In [2]:
prev = SparkSession.getActiveSession()
if prev is not None:
    prev.stop()

builder = (
    SparkSession.builder
    .appName("banking77-fraud-cluster")
    .master(MASTER)
    .config("spark.sql.session.timeZone", "UTC")          # received_ts is the user's local clock
    .config("spark.sql.shuffle.partitions", "48")         # 6 cores x 8; the default 200 is too many for this cluster
)
if not LOCAL_TEST:
    builder = (
        builder
        .config("spark.jars.packages", "org.apache.hadoop:hadoop-aws:3.5.0")
        .config("spark.executor.memory", "4g")
        .config("spark.driver.memory", "2g")
    )
spark = builder.getOrCreate()
spark.sparkContext.setLogLevel("WARN")
print("default parallelism (cores):", spark.sparkContext.defaultParallelism)

timings = {}
def timed(name, fn):
    start = time.time()
    result = fn()
    timings[name] = round(time.time() - start, 1)
    print(f"{name}: {timings[name]} s")
    return result

:: loading settings :: url = jar:file:/home/ubuntu/venv/lib/python3.14/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /home/ubuntu/.ivy2.5.2/cache
The jars for the packages stored in: /home/ubuntu/.ivy2.5.2/jars
org.apache.hadoop#hadoop-aws added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-fc6f0029-13dd-46f6-b43e-b8453ae5c5dc;1.0
    confs: [default]
    found org.apache.hadoop#hadoop-aws;3.5.0 in central
    found software.amazon.awssdk#bundle;2.35.4 in central
    found software.amazon.s3.analyticsaccelerator#analyticsaccelerator-s3;1.3.1 in central
    found org.wildfly.openssl#wildfly-openssl;2.2.5.Final in central
:: resolution report :: resolve 501ms :: artifacts dl 11ms
    :: modules in use:
    org.apache.hadoop#hadoop-aws;3.5.0 from central in [default]
    org.wildfly.openssl#wildfly-openssl;2.2.5.Final from central in [default]
    software.amazon.awssdk#bundle;2.35.4 fro

default parallelism (cores): 6

## 1 1. Raw stream as written by the generator

The generator writes one small file per chunk per day (50 chunks x 365 days), the classic *small files problem*.

In [3]:
def read_messages(path):
    df = spark.read.option("basePath", path).parquet(path)
    # Newer Spark reads received_ts as TIMESTAMP_NTZ; a plain timestamp keeps date arithmetic simple
    return df.withColumn("received_ts", F.col("received_ts").cast("timestamp"))

raw = read_messages(f"{RAW}/messages")
raw_files = len(raw.inputFiles())
raw_rows = timed("raw: count rows", lambda: raw.count())
print(f"raw files: {raw_files:,}  rows: {raw_rows:,}  partitions: {raw.rdd.getNumPartitions()}")

SLF4J: Failed to load class "org.slf4j.impl.StaticLoggerBinder".
SLF4J: Defaulting to no-operation (NOP) logger implementation
SLF4J: See http://www.slf4j.org/codes.html#StaticLoggerBinder for further details.
                                                                                

raw: count rows: 319.8 s
raw files: 18,250  rows: 50,000,000  partitions: 589

## 2 2. Compact to one file per day and write back to S3

In [4]:
def compact():
    (raw.repartition("date")              # all rows of a day land in one task -> one output file per day
        .write.mode("overwrite")
        .partitionBy("date")
        .parquet(f"{OUT}/compacted/messages"))
    spark.read.parquet(f"{RAW}/users.parquet").write.mode("overwrite").parquet(f"{OUT}/compacted/users")

timed("compact: rewrite to one file per day", compact)

compact: rewrite to one file per day: 560.9 s

In [5]:
msgs = read_messages(f"{OUT}/compacted/messages")
users = spark.read.parquet(f"{OUT}/compacted/users")
compact_files = len(msgs.inputFiles())
compact_rows = timed("compacted: count rows", lambda: msgs.count())
print(f"compacted files: {compact_files:,}  rows: {compact_rows:,}  partitions: {msgs.rdd.getNumPartitions()}")
assert compact_rows == raw_rows, "row count changed during compaction"

msgs = msgs.withColumn("group", F.coalesce("fraud_pattern", F.lit("normal")))

[Stage 11:=====================================================>  (21 + 1) / 22]

compacted: count rows: 10.0 s
compacted files: 365  rows: 50,000,000  partitions: 22

## 3 3. Part C aggregations on the full stream

In [6]:
overview = timed("C1 overview", lambda: msgs.agg(
    F.count("*").alias("messages"),
    F.countDistinct("user_id").alias("users"),
    F.countDistinct("session_id").alias("sessions"),
    F.round(F.avg(F.col("label").isNotNull().cast("int")), 4).alias("label_coverage"),
    F.round(F.avg(F.col("is_fraud").cast("int")), 4).alias("fraud_message_rate"),
    F.min("received_ts").alias("first_message"),
    F.max("received_ts").alias("last_message"),
).toPandas())
overview.to_csv(LOCAL_OUT / "C1_overview.csv", index=False)
overview.T.rename(columns={0: "value"})

C1 overview: 32.7 s

In [7]:
persona = timed("C1 persona table", lambda: users.groupBy("persona").agg(
    F.count("*").alias("users"),
    F.sum("n_messages").alias("all_messages"),
    F.sum("n_fraud_messages").alias("fraud_messages"),
).toPandas())
persona["msgs_per_user"] = (persona["all_messages"] / persona["users"]).round(1)
persona["share_of_fraud_messages"] = (persona["fraud_messages"] / persona["fraud_messages"].sum()).round(4)
persona = persona.sort_values("users", ascending=False).reset_index(drop=True)
persona.to_csv(LOCAL_OUT / "C1_persona.csv", index=False)
persona

[Stage 20:=============================>                            (1 + 1) / 2]

C1 persona table: 1.5 s

In [8]:
daily = timed("C1 daily", lambda: msgs.groupBy(F.to_date("received_ts").alias("day")).agg(
    F.count("*").alias("messages"),
    F.avg(F.col("is_fraud").cast("int")).alias("fraud_rate"),
).orderBy("day").toPandas())
daily.to_csv(LOCAL_OUT / "C1_daily.csv", index=False)
print(f"days: {len(daily)}  fraud rate: mean {daily.fraud_rate.mean():.2%}, "
      f"min {daily.fraud_rate.min():.2%}, max {daily.fraud_rate.max():.2%}")

C1 daily: 13.0 s
days: 365  fraud rate: mean 1.29%, min 1.06%, max 1.57%

### 3.1 Message gaps (window function)

`lag` over each user’s history ordered by time. This is the most expensive step: every user’s messages have to be shuffled onto one executor and sorted.

In [9]:
w = Window.partitionBy("user_id").orderBy("received_ts", "message_id")
gaps = (
    msgs.select("user_id", "session_id", "message_id", "received_ts", "group")
    .withColumn("prev_ts", F.lag("received_ts").over(w))
    .withColumn("prev_session", F.lag("session_id").over(w))
    .where(F.col("prev_ts").isNotNull())
    .withColumn("gap_s", F.col("received_ts").cast("long") - F.col("prev_ts").cast("long"))
    .withColumn("scope", F.when(F.col("session_id") == F.col("prev_session"), "within_session")
                          .otherwise("between_sessions"))
)
gaps.explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Project [user_id#50L, session_id#51L, message_id#43L, received_ts#45, group#86, prev_ts#291, prev_session#292L, (cast(received_ts#45 as bigint) - cast(prev_ts#291 as bigint)) AS gap_s#293L, CASE WHEN (session_id#51L = prev_session#292L) THEN within_session ELSE between_sessions END AS scope#294]
   +- Filter isnotnull(prev_ts#291)
      +- Window [lag(received_ts#45, -1, null) windowspecdefinition(user_id#50L, received_ts#45 ASC NULLS FIRST, message_id#43L ASC NULLS FIRST, specifiedwindowframe(RowFrame, -1, -1)) AS prev_ts#291, lag(session_id#51L, -1, null) windowspecdefinition(user_id#50L, received_ts#45 ASC NULLS FIRST, message_id#43L ASC NULLS FIRST, specifiedwindowframe(RowFrame, -1, -1)) AS prev_session#292L], [user_id#50L], [received_ts#45 ASC NULLS FIRST, message_id#43L ASC NULLS FIRST]
         +- Sort [user_id#50L ASC NULLS FIRST, received_ts#45 ASC NULLS FIRST, message_id#43L ASC NULLS FIRST], false, 0
            +- 

In [10]:
gap_q = timed("C3 gap quantiles (window function)", lambda: gaps.groupBy("scope", "group").agg(
    F.count("*").alias("n"),
    F.percentile_approx("gap_s", [0.1, 0.5, 0.9], 1000).alias("q"),
).toPandas())
gap_q[["p10_s", "median_s", "p90_s"]] = pd.DataFrame(gap_q["q"].tolist(), index=gap_q.index)
gap_q = gap_q.drop(columns="q").sort_values(["scope", "median_s"]).reset_index(drop=True)
gap_q.to_csv(LOCAL_OUT / "C3_gap_quantiles.csv", index=False)
gap_q

[Stage 36:>                                                         (0 + 1) / 1]

C3 gap quantiles (window function): 52.1 s

### 3.2 Sessions

Session table built in Spark (one row per session), written to S3 as an intermediate table, then summarised.

In [11]:
sessions = msgs.groupBy("session_id").agg(
    F.first("group").alias("group"),
    F.first("user_id").alias("user_id"),
    F.count("*").alias("n_messages"),
    ((F.max("received_ts").cast("long") - F.min("received_ts").cast("long")) / 60).alias("duration_min"),
    F.countDistinct("city").alias("n_cities"),
)
timed("C4 write session table", lambda: sessions.write.mode("overwrite").parquet(f"{OUT}/results/sessions"))
sessions = spark.read.parquet(f"{OUT}/results/sessions")

sess = timed("C4 session summary", lambda: sessions.groupBy("group").agg(
    F.count("*").alias("sessions"),
    F.percentile_approx("n_messages", 0.5).alias("median_messages"),
    F.max("n_messages").alias("max_messages"),
    F.round(F.percentile_approx("duration_min", 0.5), 1).alias("median_minutes"),
    F.round(F.percentile_approx("duration_min", 0.9), 1).alias("p90_minutes"),
).toPandas())
order = ["normal", "account_takeover", "money_mule", "card_testing", "location_anomaly", "night_activity"]
sess = sess.set_index("group").reindex(order).reset_index()
sess.to_csv(LOCAL_OUT / "C4_session_summary.csv", index=False)
sess

C4 write session table: 74.5 s

[Stage 46:>                                                         (0 + 1) / 1]

C4 session summary: 12.5 s

## 4 4. Numbers for the Spark 1-pager

In [12]:
summary = pd.Series({
    "messages": f"{raw_rows:,}",
    "users": f"{int(overview.loc[0, 'users']):,}",
    "sessions": f"{int(overview.loc[0, 'sessions']):,}",
    "raw files": f"{raw_files:,}",
    "compacted files": f"{compact_files:,}",
    "cores": spark.sparkContext.defaultParallelism,
}, name="value").to_frame()
summary.to_csv(LOCAL_OUT / "run_summary.csv")
display(summary)

timing_table = pd.Series(timings, name="seconds").to_frame()
timing_table.to_csv(LOCAL_OUT / "timings.csv")
timing_table

Before stopping: take the screenshots (Master UI on 8080, Executors and SQL / DataFrame tabs on 4040). They disappear once the session stops.

In [13]:
spark.stop()